# Quiz 2 Solutions

## Q1 — Min/Max gem using pairwise comparisons
**Goal:** Find the least and most valuable gems, using only comparisons between two gems at a time.

### Pseudocode
```text
Min-Max-Gems(gems, l, r):
    if the pile has 1 gem:
        return (that gem as MIN, that gem as MAX)

    else if the pile has 2 gems:
        compare the two gems once
        return (smaller gem as MIN, larger gem as MAX)

    else:
        Divide the pile into a left half and a right half
        (minL, maxL) ← Min-Max-Gems( left half )
        (minR, maxR) ← Min-Max-Gems( right half )

        The overall MIN is the smaller of minL and minR
        The overall MAX is the larger of maxL and maxR
        return (overall MIN, overall MAX)
```


In [ ]:
def min_max_dnc_inplace(gems: List[int], l: int, r: int):
    # one gem
    if l == r:
        x = gems[l]
        return x, x

    # two gems (one comparison)
    if r == l + 1:
        a, b = gems[l], gems[r]
        if a <= b:
            return a, b
        else:
            return b, a

    mid = (l + r) // 2
    minL, maxL = min_max_dnc_inplace(gems, l, mid)
    minR, maxR = min_max_dnc_inplace(gems, mid + 1, r)

    # combine with two comparisons
    overall_min = minL if minL <= minR else minR
    overall_max = maxL if maxL >= maxR else maxR
    return overall_min, overall_max

# Example
gems = [23, 7, 15, 42, 3, 19]
print(min_max_dnc_inplace(gems, 0, len(gems) - 1))  # expected: (3, 42)


(3, 42)


## Q2 — Longest strictly increasing contiguous subarray length
**Goal:** Given an integer array, return the length of the longest contiguous subarray that is **strictly increasing** (left to right).

### Pseudocode
```text
Max-Increasing-Run(A, l, r):
    if the segment has 1 element:
        the answer is 1

    else:
        Divide the array into A[left half] and A[right half]

        Compute the best increasing run completely inside the left half
        Compute the best increasing run completely inside the right half

        Also compute the best increasing run that crosses the middle:
            - look at the increasing run ending at the end of the left half
            - look at the increasing run starting at the beginning of the right half
            - the crossing run only connects if (last-left < first-right)

        Return the largest of:
            (best in left), (best in right), (best crossing the middle)
```


In [ ]:
def max_increase_dnc(A: List[int], l: int, r: int ):
    if l == r:
        return 1

    mid = (l + r) // 2

    best_left = max_increase_dnc(A, l, mid)
    best_right = max_increase_dnc(A, mid + 1, r)

    # Best run that crosses the boundary between mid and mid+1
    cross_best = 1
    if A[mid] < A[mid + 1]:
        # increasing suffix ending at mid
        i = mid
        left_len = 1
        while i > l and A[i - 1] < A[i]:
            left_len += 1
            i -= 1

        # increasing prefix starting at mid+1
        j = mid + 1
        right_len = 1
        while j < r and A[j] < A[j + 1]:
            right_len += 1
            j += 1

        cross_best = left_len + right_len

    return max(best_left, best_right, cross_best)

# Example
array = [1, 2, 3, 2, 1, 6, 7, 8, 9, 9, 9, 9, 1, 2, 3, 4, 5, 6, 7]
print(max_increase_dnc(array, 0, len(array) - 1))  # expected: 7


7


## Q3 — Duplicate letter in a name
**Goal:** Return `True` if any character appears more than once; otherwise return `False`.

### Pseudocode
```text
Has-Duplicate-Char(name):
    if the name has 0 or 1 character:
        there cannot be a repeated letter, so return False

    else:
        Divide the name into a left half and a right half

        Look for duplicates in the left half (recursively)
        Look for duplicates in the right half (recursively)

        If either half already contains a duplicate:
            return True

        Otherwise, check for a duplicate that occurs across the split:
            - collect the set of letters used in the left half
            - collect the set of letters used in the right half
            - if the two sets share any letter, then a duplicate exists

        If a shared letter is found:
            return True
        Otherwise:
            return False
```


In [ ]:
def has_duplicate_char_dnc(name: str, l: int, r: int):
    if l == r:
        return False, {name[l]}

    mid = (l + r) // 2

    dupL, setL = has_duplicate_char_dnc(name, l, mid)
    if dupL:
        return True, set()   # early exit

    dupR, setR = has_duplicate_char_dnc(name, mid + 1, r)
    if dupR:
        return True, set()   # early exit

    # Check for duplicates across the split (intersection)
    if len(setL) < len(setR):
        small, big = setL, setR
    else:
        small, big = setR, setL

    for ch in small:
        if ch in big:
            return True, set()

    # No cross-duplicate: union sets and return
    big.update(small)
    return False, big


# Examples
print(has_duplicate_char_dnc("Hannah", 0, len("Hannah") - 1))  # True
print(has_duplicate_char_dnc("Claire", 0, len("Claire") - 1))  # False

(True, set())
(False, {'i', 'C', 'r', 'e', 'l', 'a'})


## Q4 — Even vs odd majority
**Goal:** Return:
- `"EVEN"` if evens appear more than n/2 times
- `"ODD"` if odds appear more than n/2 times
- `"TIE"` otherwise

### Pseudocode
```text
Parity-Majority(digits):
    Divide the list into a left half and a right half
    Count (evens, odds) in the left half (recursively)
    Count (evens, odds) in the right half (recursively)
    Add the two counts to get total evens and total odds

    If total evens > n/2: return "EVEN"
    Else if total odds > n/2: return "ODD"
    Else: return "TIE"
```


In [ ]:
def parity_majority_dnc(digits: List[int], l: int = 0, r: int | None = None):
    # Top-level call: decide and return the label
    if r is None:
        n = len(digits)
        if n == 0:
            return "TIE"
        even_count, odd_count = parity_majority_dnc(digits, 0, n - 1)
        if even_count > n / 2:
            return "EVEN"
        if odd_count > n / 2:
            return "ODD"
        return "TIE"

    if l == r:
        return (1, 0) if digits[l] % 2 == 0 else (0, 1)

    mid = (l + r) // 2
    e1, o1 = parity_majority_dnc(digits, l, mid)
    e2, o2 = parity_majority_dnc(digits, mid + 1, r)
    return (e1 + e2, o1 + o2)

# Examples
print(parity_majority_dnc([2, 4, 6, 8, 1, 3, 0]))  # expected: EVEN
print(parity_majority_dnc([1, 3, 5, 7, 9, 2, 4]))  # expected: ODD
print(parity_majority_dnc([1, 2, 3, 4]))           # expected: TIE


## Quick combined test
Run the cell below to check all functions on the given examples.

In [ ]:
# Q1
assert min_max_dnc_inplace([23, 7, 15, 42, 3, 19]) == (3, 42)

# Q2
assert max_increase_dnc([1, 2, 3, 2, 1, 6, 8, 9, 10]) == 5
assert max_increase_dnc([1, 2, 3, 2, 1, 6, 7, 8, 9, 9, 9, 9, 1, 2, 3, 4, 5, 6, 7]) == 7

# Q3
assert has_duplicate_char_dnc("Hannah") is True
assert has_duplicate_char_dnc("Claire") is False

# Q4
assert parity_majority_dnc([2, 4, 6, 8, 1, 3, 0]) == "EVEN"
assert parity_majority_dnc([1, 3, 5, 7, 9, 2, 4]) == "ODD"
assert parity_majority_dnc([1, 2, 3, 4]) == "TIE"

print("All example tests passed ✅")
